# Day 18 — Fixtures

> ⚠️ **Why this matters.** Your tests have setup boilerplate — creating a WordStore, adding sample words. Fixtures factor that out. One definition, used by every test that needs it. Less typing, less drift.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/prince-curriculum/blob/main/phase-1-python-cli/lessons/18-fixtures.ipynb)

## What you'll do today

- [ ] You can write `@pytest.fixture` functions
- [ ] You understand `conftest.py` (shared fixtures)
- [ ] You know the fixture scopes: function, class, module, session
- [ ] Your test suite uses fixtures for setup

## 1. The simplest fixture

In [ ]:
# tests/test_storage.py
import pytest
from english_helper.word import Word
from english_helper.storage import WordStore

@pytest.fixture
def empty_store() -> WordStore:
    return WordStore()

def test_add_word(empty_store):
    empty_store.add(Word('thorough'))
    assert empty_store.count() == 1

**How it works:**

- `@pytest.fixture` marks a function as a fixture
- Any test function with a parameter named `empty_store` will receive the result of calling it
- pytest injects fixtures by parameter name — no decorators needed on test functions

## 2. Fixtures using fixtures

In [ ]:
@pytest.fixture
def empty_store():
    return WordStore()

@pytest.fixture
def store_with_words(empty_store):
    empty_store.add(Word('thorough', '/ˈθʌrə/'))
    empty_store.add(Word('nuance', '/ˈnuːɑːns/'))
    empty_store.add(Word('cat', '/kæt/'))
    return empty_store

def test_lookup_returns_word(store_with_words):
    w = store_with_words.lookup('thorough')
    assert w.ipa == '/ˈθʌrə/'

def test_count(store_with_words):
    assert store_with_words.count() == 3

Fixtures compose. `store_with_words` depends on `empty_store`. Each test gets a fresh chain.

> 💡 **Each test gets a fresh fixture by default** — tests don't pollute each other.

## 3. Fixture scopes

| Scope | New instance per |
|-------|------------------|
| `function` (default) | every test function |
| `class` | every test class |
| `module` | every test file |
| `session` | the whole test run |

Use larger scopes for expensive setups (DB connections, loaded model). Default `function` is safest — no cross-test contamination.

In [ ]:
@pytest.fixture(scope='session')
def slow_thing():
    print('SLOW SETUP — runs once')
    return load_500mb_dataset()

## 4. Built-in fixtures you'll use

| Fixture | Use |
|---------|-----|
| `tmp_path` | A fresh temporary `Path` per test |
| `tmp_path_factory` | For larger scopes |
| `capsys` | Capture stdout/stderr |
| `monkeypatch` | Patch env vars, attributes, etc. — auto-restored |


In [ ]:
def test_save_load(tmp_path):
    path = tmp_path / 'words.json'
    store = WordStore(path=path)
    store.add(Word('thorough'))
    store.save()
    
    loaded = WordStore.load(path=path)
    assert loaded.count() == 1

After the test, `tmp_path` is automatically cleaned up. **Use it for any test that touches files.**

## 5. Shared fixtures via `conftest.py`

Put fixtures used across multiple test files in `tests/conftest.py`:

```
tests/
├── conftest.py            # shared fixtures
├── test_word.py
├── test_storage.py
└── test_quiz.py
```

pytest auto-discovers conftest.py. Tests don't import from it; they just use the fixture by name.

In [ ]:
# tests/conftest.py
import pytest
from english_helper.word import Word
from english_helper.storage import WordStore

@pytest.fixture
def empty_store():
    return WordStore()

@pytest.fixture
def sample_words():
    return [
        Word('thorough', '/ˈθʌrə/'),
        Word('nuance', '/ˈnuːɑːns/'),
        Word('cat', '/kæt/'),
    ]

@pytest.fixture
def store_with_words(empty_store, sample_words):
    for w in sample_words:
        empty_store.add(w)
    return empty_store

Now any test in any file can use `store_with_words` — no imports needed.

## End-of-day mini-project — refactor tests to use fixtures

> 🎯 **Today's piece:** create `tests/conftest.py` with shared fixtures, refactor existing tests.

### Required fixtures in conftest.py

- `empty_store` — fresh `WordStore`
- `sample_words` — list of 3-5 representative Word instances
- `store_with_words` — populated WordStore
- `tmp_path_store` — uses tmp_path for save/load tests

### Acceptance

- All existing tests still pass.
- Test files are shorter (no duplicated setup).
- `uv run pytest -v` shows clean output.

## Connect to the project

> 🎯 **Tomorrow (Day 19):** mocks. How to test `api.py` without hitting the real internet.

**Quiz:** [18-fixtures-quiz.ipynb](18-fixtures-quiz.ipynb)